# معماری پروژه پیش‌بینی مصرف برق لوازم خانگی

**نقش:** معمار (Architect)  
**هدف:** طراحی و پیاده‌سازی Pipeline قابل دفاع برای پیش‌بینی `Appliances`

این Notebook بر اساس گزارش تحلیلگر طراحی شده و شامل:

1. اعتبارسنجی داده
2. مهندسی ویژگی‌های زمانی
3. تقسیم زمانی Train / Validation / Test
4. Baseline
5. انتخاب ویژگی با Genetic Algorithm
6. مقایسه چند خانواده مدل
7. ارزیابی با R²، MAE، RMSE و خطای Peak
8. انتخاب مدل نهایی
9. آموزش مدل نهایی و ارزیابی روی Test

> **نکته:** این Notebook عمداً از Random Train/Test Split استفاده نمی‌کند، چون داده ماهیت سری زمانی دارد.


In [ ]:
# Cell 1 — کتابخانه‌ها و تنظیمات

import warnings
warnings.filterwarnings("ignore")

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

DATA_PATH = "appliances_energy_prepared.csv"
TARGET = "Appliances"

print("Architecture environment ready.")


## Cell 2 — بارگذاری داده

فایل مورد انتظار:

`appliances_energy_prepared.csv`

گزارش تحلیلگر اعلام می‌کند دیتاست نهایی 19,735 رکورد و 31 ستون دارد و `Appliances` متغیر هدف است.


In [ ]:
# Cell 2 — Load dataset

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "CSV واقعی را کنار Notebook قرار دهید و نام آن را به appliances_energy_prepared.csv تغییر دهید."
    )

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head())
print("\nColumns:")
print(df.columns.tolist())


## Cell 3 — اعتبارسنجی اولیه

قبل از مدل‌سازی، سه مورد معماری بررسی می‌شود:

- Missing Values
- Duplicate Rows
- ترتیب زمانی

همچنین هدف باید در دیتاست وجود داشته باشد.


In [ ]:
# Cell 3 — Data validation

assert TARGET in df.columns, f"{TARGET} column is missing."

missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()

print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(duplicates))

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    print("Invalid dates:", int(df["date"].isna().sum()))
    df = df.sort_values("date").reset_index(drop=True)
    print("Chronological order verified after sorting.")
else:
    print("WARNING: date column not found. Temporal ordering cannot be independently verified.")


## Cell 4 — مهندسی ویژگی زمانی

از `date` دو ویژگی تقویمی ساخته می‌شود:

- `hour`
- `week_of_day`

این ویژگی‌ها در گزارش تحلیلگر نیز به‌عنوان ویژگی‌های مهندسی‌شده معرفی شده‌اند.

**نکته:** از ویژگی‌های آینده یا Lagهایی که باعث نشت اطلاعات شوند استفاده نمی‌کنیم.


In [ ]:
# Cell 4 — Time feature engineering

if "date" in df.columns:
    df["hour"] = df["date"].dt.hour
    df["week_of_day"] = df["date"].dt.dayofweek

    # حذف date از ورودی مدل؛ timestamp خام مستقیماً وارد مدل نمی‌شود.
    # اگر date برای گزارش لازم باشد، در df باقی می‌ماند.
else:
    raise ValueError("The dataset needs a 'date' column for the planned temporal architecture.")

display(df[["date", "hour", "week_of_day", TARGET]].head())


## Cell 5 — تعریف ویژگی‌ها و هدف

`Appliances` هدف رگرسیون است.

ویژگی‌های تصادفی `rv1` و `rv2` عمداً در مجموعه اولیه نگه داشته می‌شوند تا Genetic Algorithm بتواند نشان دهد که ویژگی‌های بدون معنای فیزیکی نباید انتخاب شوند.

`date` به‌عنوان timestamp خام وارد مدل نمی‌شود.


In [ ]:
# Cell 5 — Feature matrix

drop_from_features = {TARGET, "date"}
feature_cols = [c for c in df.columns if c not in drop_from_features]

X = df[feature_cols].copy()
y = df[TARGET].copy()

print("Number of candidate features:", len(feature_cols))
print("Target:", TARGET)
print("\nCandidate features:")
print(feature_cols)


## Cell 6 — Temporal Train / Validation / Test Split

تقسیم زمانی به شکل زیر انجام می‌شود:

- حدود 70% گذشته → Train
- حدود 10% بعدی → Validation
- حدود 20% پایانی → Test

Test فقط در مرحله نهایی استفاده می‌شود.

این ساختار از نگاه معماری برای جلوگیری از استفاده از آینده در آموزش انتخاب شده است.


In [ ]:
# Cell 6 — Temporal split

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.80)

X_train = X.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()

X_val = X.iloc[train_end:val_end].copy()
y_val = y.iloc[train_end:val_end].copy()

X_test = X.iloc[val_end:].copy()
y_test = y.iloc[val_end:].copy()

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

if "date" in df.columns:
    print("\nDate ranges:")
    print("Train:", df["date"].iloc[0], "→", df["date"].iloc[train_end-1])
    print("Validation:", df["date"].iloc[train_end], "→", df["date"].iloc[val_end-1])
    print("Test:", df["date"].iloc[val_end], "→", df["date"].iloc[-1])


## Cell 7 — Baseline

ابتدا یک مدل بسیار ساده می‌سازیم:

**پیش‌بینی = میانگین مصرف در Train**

اگر مدل‌های Machine Learning نتوانند به‌صورت معنادار از این baseline بهتر شوند، پیچیده‌تر کردن سیستم توجیه ندارد.


In [ ]:
# Cell 7 — Mean baseline

baseline_value = y_train.mean()

baseline_val_pred = np.full(len(y_val), baseline_value)
baseline_test_pred = np.full(len(y_test), baseline_value)

baseline_val = {
    "Model": "Mean Baseline",
    "R2": r2_score(y_val, baseline_val_pred),
    "MAE": mean_absolute_error(y_val, baseline_val_pred),
    "RMSE": np.sqrt(mean_squared_error(y_val, baseline_val_pred)),
}

baseline_test = {
    "Model": "Mean Baseline",
    "R2": r2_score(y_test, baseline_test_pred),
    "MAE": mean_absolute_error(y_test, baseline_test_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, baseline_test_pred)),
}

pd.DataFrame([baseline_val, baseline_test], index=["Validation", "Test"])


## Cell 8 — تابع ارزیابی

چهار معیار استفاده می‌کنیم:

- **R²:** میزان توضیح تغییرات هدف
- **MAE:** خطای مطلق متوسط
- **RMSE:** جریمه بیشتر برای خطاهای بزرگ
- **Peak MAE:** خطای MAE روی بخش پرمصرف

برای Peak، به‌جای حذف Outlierها، 10% بالایی مصرف در هر split به‌عنوان رژیم پرمصرف بررسی می‌شود.


In [ ]:
# Cell 8 — Evaluation function

def evaluate_regression(y_true, y_pred, name="Model"):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)

    peak_threshold = y_true.quantile(0.90)
    peak_mask = y_true >= peak_threshold

    return {
        "Model": name,
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Peak_MAE": mean_absolute_error(y_true[peak_mask], y_pred[peak_mask]),
        "Peak_Count": int(peak_mask.sum()),
    }


## Cell 9 — تعریف مدل‌ها

مدل‌ها از دو خانواده اصلی انتخاب شده‌اند:

**مدل‌های خطی**
- Linear Regression
- Ridge

**مدل‌های غیرخطی**
- Random Forest
- Gradient Boosting

Scaling فقط در Pipeline مدل‌های خطی استفاده می‌شود. این کار باعث می‌شود preprocessing فقط روی Train fit شود و خطر data leakage کاهش پیدا کند.


In [ ]:
# Cell 9 — Model definitions

models = {
    "Linear Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10.0))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=RANDOM_STATE
        ))
    ])
}

print(*models.keys(), sep="\n")


## Cell 10 — مقایسه مدل‌ها قبل از Feature Selection

این مرحله یک مرجع معماری می‌سازد:

**All candidate features → Model → Validation**

بعداً همین مدل‌ها با Feature Selection مقایسه می‌شوند.


In [ ]:
# Cell 10 — Full-feature model comparison

validation_results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)

    validation_results.append(
        evaluate_regression(y_val, pred_val, name)
    )

validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)

display(validation_results_df)


## Cell 11 — Genetic Algorithm

هر Chromosome مشخص می‌کند کدام ویژگی‌ها انتخاب شوند:

`1 = انتخاب شده`  
`0 = حذف شده`

Fitness:

`Validation R² - λ × feature_ratio`

بنابراین الگوریتم همزمان دو هدف دارد:

1. حفظ عملکرد پیش‌بینی
2. کاهش تعداد سنسورها/ویژگی‌ها

**نکته معماری:** Genetic Algorithm فقط روی Train/Validation تصمیم می‌گیرد و Test را نمی‌بیند.


In [ ]:
# Cell 11 — Genetic Algorithm feature selection

GA_MODEL = GradientBoostingRegressor(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=3,
    random_state=RANDOM_STATE
)

GA_IMPUTER = SimpleImputer(strategy="median")

X_train_imp = pd.DataFrame(
    GA_IMPUTER.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index
)
X_val_imp = pd.DataFrame(
    GA_IMPUTER.transform(X_val),
    columns=feature_cols,
    index=X_val.index
)

def ga_fitness(mask, feature_penalty=0.02):
    mask = np.asarray(mask, dtype=int)

    if mask.sum() == 0:
        return -np.inf

    selected = [feature_cols[i] for i, bit in enumerate(mask) if bit == 1]

    model = clone(GA_MODEL)
    model.fit(X_train_imp[selected], y_train)
    pred = model.predict(X_val_imp[selected])

    r2 = r2_score(y_val, pred)
    ratio = len(selected) / len(feature_cols)

    return r2 - feature_penalty * ratio


def tournament_selection(population, fitnesses, k=3):
    candidates = np.random.choice(len(population), size=k, replace=False)
    winner = candidates[np.argmax([fitnesses[i] for i in candidates])]
    return population[winner].copy()


def crossover(parent1, parent2):
    if len(parent1) < 2:
        return parent1.copy(), parent2.copy()

    point = np.random.randint(1, len(parent1))
    child1 = np.concatenate([parent1[:point], parent2[point:]])
    child2 = np.concatenate([parent2[:point], parent1[point:]])
    return child1, child2


def mutate(mask, mutation_rate=0.05):
    child = mask.copy()
    flips = np.random.rand(len(child)) < mutation_rate
    child[flips] = 1 - child[flips]

    if child.sum() == 0:
        child[np.random.randint(len(child))] = 1

    return child


def run_genetic_feature_selection(
    n_features,
    population_size=20,
    generations=12,
    mutation_rate=0.05,
    elite_size=2,
):
    population = np.random.randint(0, 2, size=(population_size, n_features))

    # Guarantee at least one selected feature
    for i in range(population_size):
        if population[i].sum() == 0:
            population[i, np.random.randint(n_features)] = 1

    best_mask = None
    best_fitness = -np.inf
    history = []

    for generation in range(generations):
        fitnesses = np.array([ga_fitness(m) for m in population])

        order = np.argsort(fitnesses)[::-1]
        population = population[order]
        fitnesses = fitnesses[order]

        if fitnesses[0] > best_fitness:
            best_fitness = fitnesses[0]
            best_mask = population[0].copy()

        history.append({
            "Generation": generation + 1,
            "Best_Fitness": float(fitnesses[0]),
            "Selected_Features": int(population[0].sum())
        })

        next_population = [population[i].copy() for i in range(elite_size)]

        while len(next_population) < population_size:
            p1 = tournament_selection(population, fitnesses)
            p2 = tournament_selection(population, fitnesses)

            c1, c2 = crossover(p1, p2)

            c1 = mutate(c1, mutation_rate)
            c2 = mutate(c2, mutation_rate)

            next_population.append(c1)
            if len(next_population) < population_size:
                next_population.append(c2)

        population = np.array(next_population)

    return best_mask, pd.DataFrame(history)


best_mask, ga_history = run_genetic_feature_selection(len(feature_cols))

selected_features = [
    feature_cols[i] for i, bit in enumerate(best_mask) if bit == 1
]

print("Selected features:")
print(selected_features)

print("\nNumber selected:", len(selected_features))
display(ga_history.tail())


## Cell 12 — بررسی نتیجه Feature Selection

در این مرحله بررسی می‌کنیم آیا Genetic Algorithm واقعاً توانسته ویژگی‌های غیرضروری را کنار بگذارد.

به‌خصوص `rv1` و `rv2` باید از نظر عملکردی ارزش کمی نشان دهند؛ این دو متغیر در گزارش به‌عنوان نویز مصنوعی معرفی شده‌اند.


In [ ]:
# Cell 12 — GA history and selected feature visualization

plt.figure(figsize=(8, 4))
plt.plot(
    ga_history["Generation"],
    ga_history["Best_Fitness"],
    marker="o"
)
plt.xlabel("Generation")
plt.ylabel("Best Fitness")
plt.title("Genetic Algorithm Progress")
plt.grid(alpha=0.25)
plt.show()

print("Selected features:")
for f in selected_features:
    print(" -", f)


## Cell 13 — مقایسه مدل‌ها با Feature Selection

اکنون دقیقاً همان مدل‌ها را با زیرمجموعه منتخب Genetic Algorithm اجرا می‌کنیم.

این مقایسه نشان می‌دهد آیا کاهش تعداد ویژگی‌ها باعث حفظ یا بهبود عملکرد شده است یا خیر.


In [ ]:
# Cell 13 — Models with selected features

selected_validation_results = []

for name, model in models.items():
    model_fs = clone(model)
    model_fs.fit(X_train[selected_features], y_train)

    pred_val = model_fs.predict(X_val[selected_features])

    selected_validation_results.append(
        evaluate_regression(
            y_val,
            pred_val,
            name + " + GA Features"
        )
    )

selected_validation_results_df = (
    pd.DataFrame(selected_validation_results)
    .sort_values("R2", ascending=False)
    .reset_index(drop=True)
)

display(selected_validation_results_df)


## Cell 14 — جدول نهایی مقایسه معماری

معیار انتخاب مدل:

1. عملکرد Validation
2. MAE و RMSE
3. عملکرد Peak
4. تعداد ویژگی‌ها
5. پیچیدگی مدل

**Test هنوز برای انتخاب مدل استفاده نمی‌شود.**


In [ ]:
# Cell 14 — Architecture decision table

architecture_results = selected_validation_results_df.copy()

architecture_results["Feature_Count"] = len(selected_features)

architecture_results = architecture_results[
    [
        "Model",
        "R2",
        "MAE",
        "RMSE",
        "Peak_MAE",
        "Feature_Count"
    ]
]

display(architecture_results)


## Cell 15 — انتخاب مدل نهایی

مدل نهایی از Validation انتخاب می‌شود.

برای جلوگیری از انتخاب مدلی که فقط به خاطر R² بالا در شرایط عادی مناسب است، Peak_MAE نیز در تصمیم معماری گزارش می‌شود.

در این مرحله هنوز Test دیده نشده است.


In [ ]:
# Cell 15 — Select final architecture

# Primary ranking by validation R².
# Secondary metrics are retained for engineering judgment.
best_row = architecture_results.sort_values(
    ["R2", "RMSE", "Peak_MAE"],
    ascending=[False, True, True]
).iloc[0]

best_model_name = best_row["Model"].replace(" + GA Features", "")

print("Selected final model:", best_model_name)
print("Selected features:", selected_features)
print("Validation R²:", best_row["R2"])
print("Validation MAE:", best_row["MAE"])
print("Validation RMSE:", best_row["RMSE"])
print("Validation Peak MAE:", best_row["Peak_MAE"])


## Cell 16 — آموزش مدل نهایی و ارزیابی روی Test

پس از انتخاب مدل، Train و Validation با هم ترکیب می‌شوند.

سپس مدل نهایی فقط یک بار روی Test ارزیابی می‌شود.

این Test نماینده بخش زمانی آینده نسبت به Train/Validation است.


In [ ]:
# Cell 16 — Final fit and untouched test evaluation

final_model = clone(models[best_model_name])

X_train_val = pd.concat(
    [X_train[selected_features], X_val[selected_features]],
    axis=0
)
y_train_val = pd.concat([y_train, y_val], axis=0)

final_model.fit(X_train_val, y_train_val)

test_pred = final_model.predict(X_test[selected_features])

final_test_result = evaluate_regression(
    y_test,
    test_pred,
    best_model_name + " + GA Features — FINAL TEST"
)

final_test_df = pd.DataFrame([final_test_result])
display(final_test_df)


## Cell 17 — تحلیل خطا و Peak

در این بخش مدل را در دو رژیم بررسی می‌کنیم:

- مصرف عادی
- مصرف پرمصرف / Peak

هدف این است که یک R² یا MAE خوب، ضعف مدل در ساعات/رخدادهای مهم را پنهان نکند.


In [ ]:
# Cell 17 — Error analysis

error_analysis = pd.DataFrame({
    "Actual": y_test.reset_index(drop=True),
    "Predicted": pd.Series(test_pred).reset_index(drop=True)
})

error_analysis["Error"] = error_analysis["Actual"] - error_analysis["Predicted"]
error_analysis["Absolute_Error"] = error_analysis["Error"].abs()

peak_threshold = error_analysis["Actual"].quantile(0.90)
error_analysis["Regime"] = np.where(
    error_analysis["Actual"] >= peak_threshold,
    "Peak",
    "Normal"
)

regime_summary = (
    error_analysis
    .groupby("Regime")
    .agg(
        Count=("Actual", "size"),
        MAE=("Absolute_Error", "mean"),
        RMSE=("Error", lambda x: np.sqrt(np.mean(x**2))),
        Mean_Actual=("Actual", "mean"),
        Mean_Predicted=("Predicted", "mean")
    )
)

display(regime_summary)


## Cell 18 — نمودار Actual vs Predicted

نزدیک بودن نقاط به خط ایده‌آل نشان می‌دهد پیش‌بینی‌ها به مقدار واقعی نزدیک هستند.

در یک پروژه انرژی، پراکندگی زیاد در مقادیر Peak باید جداگانه بررسی شود.


In [ ]:
# Cell 18 — Actual vs Predicted

plt.figure(figsize=(7, 6))
plt.scatter(y_test, test_pred, alpha=0.35)

line_min = min(y_test.min(), np.min(test_pred))
line_max = max(y_test.max(), np.max(test_pred))

plt.plot(
    [line_min, line_max],
    [line_min, line_max],
    linestyle="--"
)

plt.xlabel("Actual Appliances (Wh)")
plt.ylabel("Predicted Appliances (Wh)")
plt.title("Final Model — Actual vs Predicted")
plt.grid(alpha=0.2)
plt.show()


## Cell 19 — ذخیره Feature Selection و نتایج

برای استفاده در گزارش نهایی، دو فایل خروجی ساخته می‌شود:

- `selected_features.csv`
- `final_test_results.csv`


In [ ]:
# Cell 19 — Export architecture outputs

pd.DataFrame({
    "Selected_Feature": selected_features
}).to_csv("selected_features.csv", index=False)

final_test_df.to_csv("final_test_results.csv", index=False)

print("Saved:")
print("- selected_features.csv")
print("- final_test_results.csv")


# جمع‌بندی معماری

### تصمیم‌های معماری این پروژه

1. مسئله یک **Regression** روی `Appliances` است.
2. به دلیل ماهیت زمانی، Split به‌صورت **Temporal** انجام می‌شود.
3. یک **Baseline** قبل از مدل‌های پیچیده ساخته می‌شود.
4. Feature Selection با **Genetic Algorithm** انجام می‌شود.
5. مدل‌های خطی و غیرخطی با یک معیار مشترک مقایسه می‌شوند.
6. Scaling فقط در Pipeline مدل‌هایی که به آن نیاز دارند انجام می‌شود.
7. Outlierهای واقعی حذف نمی‌شوند؛ Peak به‌عنوان یک رژیم مهم ارزیابی می‌شود.
8. Test فقط در پایان و پس از انتخاب معماری دیده می‌شود.
9. معیارهای اصلی: **R²، MAE، RMSE و Peak MAE**.
10. مدل نهایی باید علاوه بر دقت، از نظر تعداد ویژگی و پیچیدگی نیز قابل دفاع باشد.

### نکته تحویل

این Notebook «معماری» است و نباید قبل از اجرای CSV واقعی، عددی برای بهترین مدل یا ویژگی‌های منتخب ادعا کند.
